# Dental AI — Variant 2: Rýchly tréning (YOLOv8n, subset)

- Záložná/rychlá variantá: ~1,5–2 h na T4/P100
- YOLOv8n, imgsz 512, 40 epók, vyvážený subset (max 4 000 obrázkov)
- Slúži na rýchlu iteráciu a overenie pipeline; V1 dáva vyššie skóre

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print(ultralytics.__version__, torch.__version__, torch.cuda.is_available())
!df -h /kaggle/working 2>/dev/null | head -3

In [ ]:
import os, glob, yaml, random, shutil
random.seed(42)
SRC = '/kaggle/input/mega-dataset-dental'
hits = glob.glob(f'{SRC}/**/data.yaml', recursive=True)
root = os.path.dirname(hits[0])
print('root:', root)

# Vyvazeny subset: max 4000 train, vsetky valid (pre stabilne porovnanie)
MAX_TRAIN = 4000
W = '/kaggle/working/ds'
for split, dst in [('train','train'), ('valid','valid'), ('test','test')]:
    imgs = sorted(os.listdir(os.path.join(root, split, 'images')))
    if split == 'train':
        imgs = random.sample(imgs, min(MAX_TRAIN, len(imgs)))
    os.makedirs(f'{W}/{dst}/images', exist_ok=True)
    os.makedirs(f'{W}/{dst}/labels', exist_ok=True)
    for im in imgs:
        shutil.copy(os.path.join(root, split, 'images', im), f'{W}/{dst}/images/{im}')
        lbl = os.path.join(root, split, 'labels', im.rsplit('.',1)[0] + '.txt')
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{W}/{dst}/labels/')
    print(dst, len(os.listdir(f'{W}/{dst}/images')))

cfg = yaml.safe_load(open(os.path.join(root,'data.yaml')))
cfg.update({'path': W, 'train':'train/images', 'val':'valid/images', 'test':'test/images'})
yaml.safe_dump(cfg, open(f'{W}/data.yaml','w'), sort_keys=False)
print(open(f'{W}/data.yaml').read())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
model.train(
    data=f'{W}/data.yaml',
    epochs=40, imgsz=512, batch=32,
    patience=10, optimizer='AdamW', lr0=0.001, cos_lr=True,
    cache='ram', hsv_v=0.5, mosaic=1.0, mixup=0.1,
    project='/kaggle/working/train', name='v2_yolov8n_fast', exist_ok=True,
)

In [ ]:
best = YOLO('/kaggle/working/train/v2_yolov8n_fast/weights/best.pt')
metrics = best.val(data=f'{W}/data.yaml', split='test')
print(metrics.results_dict)
import shutil
shutil.copy('/kaggle/working/train/v2_yolov8n_fast/weights/best.pt', '/kaggle/working/best_v2.pt')
shutil.copy('/kaggle/working/train/v2_yolov8n_fast/results.csv', '/kaggle/working/results_v2.csv')
shutil.rmtree('/kaggle/working/train', ignore_errors=True)
shutil.rmtree(W, ignore_errors=True)
!ls -la /kaggle/working